<a href="https://colab.research.google.com/github/lucasbomfim1207-ai/CP2_MLAM/blob/main/notebook_regressao_pib_abcr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regressão linear: PIB × Índice ABCR de fluxo de veículos (Brasil, 2006–2025)

**Pergunta:** existe relação entre a atividade econômica brasileira (índice de volume do PIB) e o fluxo de veículos nas rodovias pedagiadas (Índice ABCR)? Um modelo de regressão linear simples consegue estimar o ABCR a partir do PIB?

## 1. Fontes consultadas

| Indicador | Fonte | Arquivo no repositório | Detalhe |
| --- | --- | --- | --- |
| Índice de volume do PIB | IBGE, [Tabela 1620 do SIDRA](https://sidra.ibge.gov.br/tabela/1620) (Contas Nacionais Trimestrais) | `dados/tabela1620.xlsx` e `dados/tabela1620.csv` | Série encadeada, **sem ajuste sazonal**, base: média 1995 = 100. Brasil, "PIB a preços de mercado", 1º tri/2006 a 4º tri/2025. |
| Índice ABCR | ABCR, [Índice ABCR](https://melhoresrodovias.org.br/indice-abcr_2/) (edição de agosto/2026) | `dados/abcr_0826.xlsx` | Aba `(C) Original` (série original, **sem ajuste sazonal**), bloco **Brasil**, coluna **TOTAL** (leves + pesados), base 1999 = 100. |

> As duas séries têm bases diferentes (1995 = 100 e 1999 = 100). Isso não afeta a correlação, mas significa que o coeficiente angular da regressão não é uma "elasticidade" diretamente interpretável.

In [1]:
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.options.display.float_format = '{:.2f}'.format
ANO_INI, ANO_FIM = 2006, 2025

## 2. Organização da base

### 2.1 PIB: média dos 4 índices trimestrais de cada ano

In [6]:
raw = pd.read_excel('dados/tabela1620.xlsx', sheet_name='Tabela', header=None)

# linha 3: rótulos dos trimestres; linha 5: valores de "Brasil"
pib_tri = pd.DataFrame({
    'Trimestre': raw.iloc[3, 1:].astype(str).values,
    'PIB': raw.iloc[5, 1:].astype(float).values,
})
pib_tri['Ano'] = pib_tri['Trimestre'].str[-4:].astype(int)

# Verificação cruzada: o CSV traz exatamente os mesmos valores do XLSX?
with open('dados/tabela1620.csv', encoding='utf-8-sig', newline='') as f:
    linhas = list(csv.reader(f, delimiter=';'))
linha_brasil = next(l for l in linhas if l and l[0] == 'Brasil')
valores_csv = [float(v.replace(',', '.')) for v in linha_brasil[1:] if v != '']
print('CSV e XLSX idênticos:', np.allclose(valores_csv, pib_tri['PIB'].values))

print('Trimestres por ano:', sorted(pib_tri.groupby('Ano').size().unique()))
pib_anual = pib_tri.groupby('Ano')['PIB'].mean().rename('PIB_indice')
print(pib_anual.head(3).to_string())

FileNotFoundError: [Errno 2] No such file or directory: 'dados/tabela1620.xlsx'

### 2.2 ABCR: média dos 12 índices mensais de cada ano

In [ ]:
abcr_raw = pd.read_excel('dados/abcr_0826.xlsx', sheet_name='(C) Original', header=None, skiprows=3)

# coluna 0 = data; coluna 3 = Brasil / TOTAL
abcr_m = abcr_raw.iloc[:, [0, 3]].copy()
abcr_m.columns = ['Data', 'ABCR']
abcr_m['Data'] = pd.to_datetime(abcr_m['Data'], errors='coerce')
abcr_m = abcr_m.dropna()
abcr_m = abcr_m[abcr_m['Data'].dt.year.between(ANO_INI, ANO_FIM)]

meses = abcr_m.groupby(abcr_m['Data'].dt.year).size()
print('Meses por ano:', sorted(meses.unique()))
abcr_anual = abcr_m.groupby(abcr_m['Data'].dt.year)['ABCR'].mean().rename('ABCR_indice')
print(abcr_anual.head(3).to_string())

### 2.3 Tabela final (uma linha por ano, os dois indicadores no mesmo ano)

In [ ]:
df = pd.concat([pib_anual, abcr_anual], axis=1).loc[ANO_INI:ANO_FIM]
df.index.name = 'Ano'
df = df.reset_index()

# Conferências: 20 anos, sem faltantes, anos consecutivos
assert len(df) == 20 and df.isna().sum().sum() == 0
assert (df['Ano'].diff().dropna() == 1).all()

df.to_csv('dados/base_pib_abcr.csv', index=False)
print(df.to_string(index=False))

## 3. Análise da relação

### 3.1 Dispersão e correlação

In [ ]:
corr = df['PIB_indice'].corr(df['ABCR_indice'])
print(f'Correlação de Pearson (PIB_indice x ABCR_indice): {corr:.4f}')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df['PIB_indice'], df['ABCR_indice'], color='tab:blue')
for _, r in df.iterrows():
    ax.annotate(int(r['Ano']), (r['PIB_indice'], r['ABCR_indice']),
                textcoords='offset points', xytext=(4, 4), fontsize=7)
ax.set_xlabel('Índice de volume do PIB (média 1995 = 100)')
ax.set_ylabel('Índice ABCR - fluxo total (1999 = 100)')
ax.set_title('PIB x Índice ABCR, 2006-2025')
ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

**Descrição.** A correlação de Pearson é de aproximadamente **0,96**, ou seja, uma relação positiva e muito forte: nos anos em que o PIB está mais alto, o fluxo de veículos também tende a estar. A nuvem de pontos é aproximadamente linear, com duas particularidades visíveis: o ano de **2020** (pandemia) fica bem abaixo da reta, porque o fluxo de veículos caiu muito mais do que o PIB, e o período **2014–2019** forma um grupo em que o PIB oscila pouco e o ABCR também.

**Cuidado:** as duas séries têm tendência de crescimento ao longo do tempo. Parte dessa correlação alta pode vir apenas de ambas "subirem com o tempo", e correlação não demonstra causa e efeito.

## 4. Treinamento do modelo

- Entrada `X`: `PIB_indice`; alvo `y`: `ABCR_indice`.
- Treino: primeiros 16 anos (2006–2021). Teste: últimos 4 anos (2022–2025).
- Ordem cronológica mantida, sem embaralhar. O teste é, portanto, uma "previsão para o futuro" em relação ao treino.

In [ ]:
treino, teste = df.iloc[:16], df.iloc[16:]
X_treino, y_treino = treino[['PIB_indice']], treino['ABCR_indice']
X_teste,  y_teste  = teste[['PIB_indice']],  teste['ABCR_indice']
print('Treino:', treino['Ano'].min(), '-', treino['Ano'].max(), f'({len(treino)} anos)')
print('Teste: ', teste['Ano'].min(),  '-', teste['Ano'].max(),  f'({len(teste)} anos)')

modelo = LinearRegression()
modelo.fit(X_treino, y_treino)

print(f'\nABCR = {modelo.intercept_:.3f} + {modelo.coef_[0]:.4f} x PIB')
print(f'R² no treino: {modelo.score(X_treino, y_treino):.4f}')

## 5. Avaliação no conjunto de teste

In [ ]:
y_prev = modelo.predict(X_teste)

mae = mean_absolute_error(y_teste, y_prev)
mse = mean_squared_error(y_teste, y_prev)
r2  = r2_score(y_teste, y_prev)
print(f'MAE : {mae:.3f}')
print(f'MSE : {mse:.3f}  (RMSE = {np.sqrt(mse):.3f})')
print(f'R²  : {r2:.3f}')

resultado = teste[['Ano', 'PIB_indice']].copy()
resultado['ABCR_observado'] = y_teste.values
resultado['ABCR_previsto'] = y_prev
resultado['Erro (prev - obs)'] = resultado['ABCR_previsto'] - resultado['ABCR_observado']
resultado['Erro %'] = 100 * resultado['Erro (prev - obs)'] / resultado['ABCR_observado']
print()
print(resultado.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(treino['PIB_indice'], treino['ABCR_indice'], label='Treino (2006-2021)', color='tab:blue')
ax.scatter(teste['PIB_indice'], teste['ABCR_indice'], label='Teste observado (2022-2025)', color='tab:green', marker='s')
ax.scatter(teste['PIB_indice'], y_prev, label='Teste previsto', color='tab:red', marker='x', s=60)
xs = pd.DataFrame({'PIB_indice': np.linspace(df['PIB_indice'].min(), df['PIB_indice'].max(), 50)})
ax.plot(xs, modelo.predict(xs), color='black', lw=1, label='Reta ajustada (treino)')
ax.set_xlabel('PIB_indice'); ax.set_ylabel('ABCR_indice')
ax.set_title('Regressão linear: treino, teste e previsões')
ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 5.1 Significado das métricas

- **MAE (erro absoluto médio):** média dos erros em valor absoluto, na mesma unidade do ABCR (pontos de índice). Um MAE de ≈ 4,9 significa que, em média, a previsão erra cerca de 5 pontos, o que é perto de **3%** do nível do índice no período de teste (≈ 165).
- **MSE (erro quadrático médio):** média dos erros ao quadrado (≈ 26). Penaliza erros grandes e está em "pontos²", por isso é mais fácil ler a raiz (RMSE ≈ 5,1 pontos).
- **R²:** fração da variação do teste explicada pelo modelo, em comparação com simplesmente prever a média do teste. Aqui, R² ≈ 0,48. Um R² de 1 seria perfeito; 0 equivale a prever a média.

### 5.2 O modelo estimou valores próximos dos reais?

**Em termos relativos, sim; em termos de "acompanhar a variação entre os anos", só em parte.**

- Os erros são pequenos em relação ao nível do índice (entre ≈ 2% e ≈ 4%), e o modelo acertou a direção: o ABCR sobe de 2022 a 2025, assim como o previsto.
- Porém, o modelo **superestimou os quatro anos** (erro sempre positivo). Isso indica um viés sistemático: no período recente, o fluxo de veículos cresceu menos por ponto de PIB do que a relação média 2006–2021 sugeria. O R² moderado reflete isso: no teste, o ABCR varia só de ≈ 154 a ≈ 173, então um desvio constante de ≈ 5 pontos pesa muito em relação a essa pequena variação.
- Com apenas 4 pontos de teste, o R² é uma métrica instável; não deve ser lida como "o modelo é ruim" nem como "o modelo é bom".

## 6. Conclusão

1. Existe uma **relação positiva muito forte** entre o índice de volume do PIB e o Índice ABCR (correlação ≈ 0,96 em 2006–2025): quando a atividade econômica cresce, o fluxo de veículos nas rodovias pedagiadas tende a crescer junto.
2. A regressão linear simples (ABCR ≈ −56,8 + 1,21 × PIB, ajustada em 2006–2021) produziu estimativas **próximas dos valores reais** (MAE ≈ 4,9; erro entre ≈ 2% e 4%), mas com **viés para cima** nos anos de teste (R² ≈ 0,48), sugerindo que a relação mudou um pouco no período recente.
3. **Limitações:** (i) só 20 observações anuais e 4 de teste; (ii) as duas séries crescem com o tempo, o que infla a correlação de níveis; (iii) a pandemia de 2020 é um ponto atípico dentro do treino; (iv) o ABCR cobre rodovias pedagiadas, não todo o país; (v) o PIB mede produção, enquanto o ABCR depende também de preço de combustível, calendário, composição entre veículos leves e pesados etc.
4. **Correlação alta não demonstra causa e efeito.** Os resultados mostram associação estatística; para investigar causalidade seria preciso, por exemplo, trabalhar com variações (taxas de crescimento), incluir outras variáveis e usar séries mais longas ou de maior frequência.

## 7. Dificuldades encontradas e soluções adotadas

| Dificuldade | Solução |
| --- | --- |
| A planilha da ABCR tem várias abas (gráficos, quadros, série original, série dessazonalizada) e vários blocos de colunas (Brasil, São Paulo, Paraná, Rio de Janeiro...). | Usei a aba `(C) Original` (sem ajuste sazonal, como pedido) e o bloco **Brasil / TOTAL**, e conferi os cabeçalhos antes de selecionar as colunas. |
| A aba `(D) Dessazonalizado` também traz "números-índice" e poderia ser usada por engano. | Descartada: o enunciado pede séries sem ajuste sazonal. |
| O CSV da Tabela 1620 tem linhas de cabeçalho com número de colunas diferente, o que quebra `pd.read_csv`. | Li o XLSX com `pandas` e usei o módulo `csv` só para **validar** que o CSV traz os mesmos valores. |
| O PIB é trimestral (4 valores/ano) e o ABCR é mensal (12 valores/ano). | Agreguei ambos pela média anual e verifiquei com `assert` que todos os anos têm 4 trimestres / 12 meses. |
| A planilha ABCR vai até agosto/2026, mas o PIB só até 2025. | Restringi a 2006–2025, os 20 anos completos em comum. |
| Bases diferentes (1995 = 100 e 1999 = 100). | Mantive os números-índice originais, como pede o enunciado, e registrei que isso impede interpretar o coeficiente como elasticidade. |